# Context-aware BERT – Nhận diện cảm xúc trong hội thoại (Nhóm 9)

**Yêu cầu:** Bật GPU trước khi chạy: `Runtime` → `Change runtime type` → **T4 GPU** → `Save`.

In [1]:
# 1. Clone repo và cài dependencies
%cd /content
!rm -rf /content/project_nhom9
!git clone https://github.com/trangdx2602/bert-context.git /content/project_nhom9
%cd /content/project_nhom9/Codebase

!pip install -r requirements.txt -q

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("LỖI: Chưa bật GPU trên Google Colab.")

/content
Cloning into '/content/project_nhom9'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 80 (delta 36), reused 61 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 4.22 MiB | 38.88 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/project_nhom9/Codebase
GPU: Tesla T4


## Upload dữ liệu MELD

Chạy cell bên dưới và upload 3 file CSV:
- `train_sent_emo.csv`
- `val_sent_emo.csv`
- `test_sent_emo.csv`

In [2]:
# 2. Upload file dữ liệu MELD
import os
from google.colab import files

os.makedirs('/content/project_nhom9/Documents', exist_ok=True)

print("Upload 3 file: train_sent_emo.csv, val_sent_emo.csv, test_sent_emo.csv")
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = f'/content/project_nhom9/Documents/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"  Đã lưu: {dest}")

Upload 3 file: train_sent_emo.csv, val_sent_emo.csv, test_sent_emo.csv


Saving test_sent_emo.csv to test_sent_emo.csv
Saving train_sent_emo.csv to train_sent_emo.csv
Saving val_sent_emo.csv to val_sent_emo.csv
  Đã lưu: /content/project_nhom9/Documents/test_sent_emo.csv
  Đã lưu: /content/project_nhom9/Documents/train_sent_emo.csv
  Đã lưu: /content/project_nhom9/Documents/val_sent_emo.csv


## Huấn luyện – Ablation context window (k = 1, 3, 5)

Chạy tuần tự 3 cell bên dưới để so sánh ảnh hưởng của độ dài context.

Sau khi train xong, có thể mở TensorBoard để xem loss, accuracy, weighted F1 và thời điểm checkpoint tốt nhất.

In [3]:
# 3a. Huấn luyện với k=1 (1 câu trước + câu hiện tại)
!python train.py --model bert_context --context_k 1 --batch_size 32 --num_workers 2 --log_dir runs/bert_context_k1


  Model            : bert_context
  Context k        : 1
  Device           : cuda
  Mixed Precision  : ON
  Batch size       : 32  (effective: 32)
  Accum steps      : 1
  Epochs           : 8

[1/4] Đang pre-tokenize và load dữ liệu...
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 190kB/s]
vocab.txt: 232kB [00:00, 9.90MB/s]
tokenizer.json: 466kB [00:00, 5.56MB/s]
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Train: 9,989 samples
  Val  : 1,109 samples

[2/4] Class weights: ['0.303', '1.184', '5.325', '2.089', '0.819', '5.266', '1.287']

[3/4] Load model...
config.json: 100% 570/570 [00:00<00:00, 2.93MB/s]
model.safetensors: 100% 440M/440M [00:02<00:00, 198MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 936.66it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight               

In [4]:
# 3b. Huấn luyện với k=3 (3 câu trước + câu hiện tại)
!python train.py --model bert_context --context_k 3 --batch_size 32 --num_workers 2 --log_dir runs/bert_context_k3


  Model            : bert_context
  Context k        : 3
  Device           : cuda
  Mixed Precision  : ON
  Batch size       : 32  (effective: 32)
  Accum steps      : 1
  Epochs           : 8

[1/4] Đang pre-tokenize và load dữ liệu...
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Train: 9,989 samples
  Val  : 1,109 samples

[2/4] Class weights: ['0.303', '1.184', '5.325', '2.089', '0.819', '5.266', '1.287']

[3/4] Load model...
Loading weights: 100% 199/199 [00:00<00:00, 926.52it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weigh

In [5]:
# 3c. Huấn luyện với k=5 (5 câu trước + câu hiện tại)
!python train.py --model bert_context --context_k 5 --batch_size 32 --num_workers 2 --log_dir runs/bert_context_k5


  Model            : bert_context
  Context k        : 5
  Device           : cuda
  Mixed Precision  : ON
  Batch size       : 32  (effective: 32)
  Accum steps      : 1
  Epochs           : 8

[1/4] Đang pre-tokenize và load dữ liệu...
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Train: 9,989 samples
  Val  : 1,109 samples

[2/4] Class weights: ['0.303', '1.184', '5.325', '2.089', '0.819', '5.266', '1.287']

[3/4] Load model...
Loading weights: 100% 199/199 [00:00<00:00, 956.03it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias 

## TensorBoard

Sau khi chạy xong các cell train, chạy 2 cell dưới đây để xem đồ thị train/val loss, accuracy, weighted F1 và thời điểm checkpoint tốt nhất.

In [ ]:
# 3d. Bat TensorBoard trong Colab
%load_ext tensorboard

In [ ]:
# 3e. Hien thi TensorBoard logs
%tensorboard --logdir /content/project_nhom9/Codebase/runs

## Đánh giá trên tập test

In [ ]:
# 4a. Đánh giá k=1
!python evaluate.py --model bert_context --context_k 1 --num_workers 2


Device: cuda  |  AMP: ON
  Đang pre-tokenize test...
Test set: 2,610 samples

Loading weights: 100% 199/199 [00:00<00:00, 1431.92it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  [Checkpoint loaded] epoch=2, val_f1=0.5616
Evaluating: 100% 82/82 [00:04<00:00, 18.87it/s]

  Mo

In [ ]:
# 4b. Đánh giá k=3
!python evaluate.py --model bert_context --context_k 3 --num_workers 2


Device: cuda  |  AMP: ON
  Đang pre-tokenize test...
Test set: 2,610 samples

Loading weights: 100% 199/199 [00:00<00:00, 894.31it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  [Checkpoint loaded] epoch=2, val_f1=0.5630
Evaluating: 100% 82/82 [00:04<00:00, 18.77it/s]

  Mod

In [ ]:
# 4c. Đánh giá k=5
!python evaluate.py --model bert_context --context_k 5 --num_workers 2


Device: cuda  |  AMP: ON
  Đang pre-tokenize test...
Test set: 2,610 samples

Loading weights: 100% 199/199 [00:00<00:00, 1303.88it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  [Checkpoint loaded] epoch=4, val_f1=0.5663
Evaluating: 100% 82/82 [00:04<00:00, 18.94it/s]

  Mo

## Tổng hợp kết quả

| Mô hình | Context k | Accuracy | Weighted F1 |
|---------|-----------|----------|-------------|
| Context-aware BERT | k=1 | 0.5471 | 0.5715 |
| Context-aware BERT | k=3 | 0.5429 | 0.5643 |
| Context-aware BERT | k=5 | **0.5678** | **0.5819** |

**Nhận xét:** k=5 đạt kết quả tốt nhất (F1=0.5819). Xu hướng không hoàn toàn đơn điệu do k=3 thấp hơn k=1, có thể do chuỗi 3 câu bị truncate với MAX_LEN=128. Nhãn `neutral` nhận diện tốt nhất, `fear` và `disgust` kém nhất do số lượng mẫu quá ít.

## Bi?u ?? minh ho? k?t qu?

Hai cell d??i ??y gi?p hi?n th? tr?c quan k?t qu? th? nghi?m v? ph?n b? nh?n c?a t?p train ?? h? tr? thuy?t tr?nh.


In [ ]:
# 5a. V? bi?u ?? Accuracy v? Weighted F1 theo context_k
import matplotlib.pyplot as plt

context_k = [1, 3, 5]
accuracy = [0.5471, 0.5429, 0.5678]
weighted_f1 = [0.5715, 0.5643, 0.5819]

x = range(len(context_k))
width = 0.35

plt.figure(figsize=(8, 5))
bars1 = plt.bar([i - width/2 for i in x], accuracy, width=width, label='Accuracy', color='#4C78A8')
bars2 = plt.bar([i + width/2 for i in x], weighted_f1, width=width, label='Weighted F1', color='#F58518')

plt.xticks(list(x), [f'k={k}' for k in context_k])
plt.ylim(0.50, 0.60)
plt.xlabel('Context window')
plt.ylabel('Score')
plt.title('So s?nh Accuracy v? Weighted F1 theo context_k')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.3)

for bars in (bars1, bars2):
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, h + 0.001, f'{h:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# 5b. V? ph?n b? nh?n tr?n t?p train
import pandas as pd
import matplotlib.pyplot as plt

train_df = pd.read_csv('/content/project_nhom9/Documents/train_sent_emo.csv')
label_counts = train_df['Emotion'].value_counts()
order = ['neutral', 'joy', 'surprise', 'anger', 'sadness', 'disgust', 'fear']
label_counts = label_counts.reindex(order)

plt.figure(figsize=(9, 5))
bars = plt.bar(label_counts.index, label_counts.values, color='#72B7B2')
plt.title('Ph?n b? nh?n c?m x?c tr?n t?p train')
plt.xlabel('Emotion label')
plt.ylabel('S? l??ng m?u')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.xticks(rotation=20)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 40, f'{int(h)}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()
